# nb_nee_fallback_analyzer — measure NEE vs JVM instead of assuming
**The question this answers:** is the Native Execution Engine actually accelerating *your* queries,
or is it enabled-but-inactive while you pay conversion overhead? Section 25 of the internals doc
explains the mechanics; this notebook produces the evidence.

**What it does:** runs a representative query set twice (NEE on / NEE off), captures wall-clock and
plan composition (`*Transformer` / `*NativeFileScan` vs `VeloxColumnarToRowExec`), computes native
coverage per query, and — on Runtime 2.0 — quantifies the **ANSI × NEE trade-off** on your data.

**Where it runs:** NEE is a Fabric feature. Locally this notebook exercises all its logic against
JVM Spark and labels the NEE-specific comparison as Fabric-only — every measurement cell is honest
about which mode it actually ran in.

In [1]:
NOTEBOOK_NAME = "nb_nee_fallback_analyzer"
TABLES_ROOT   = "/tmp/fabric_nee_demo"
REPORT_PATH   = f"{TABLES_ROOT}/_ops/nee_report"
RUNTIME       = "auto"      # auto | fabric-1.3 | fabric-2.0
REPEATS       = 1           # repeat each query to reduce noise (take the min)

In [2]:
import os, re, time, json, statistics
from datetime import datetime, timezone

def detect():
    try:
        import notebookutils  # noqa
        fab = True
    except ImportError:
        fab = False
    import pyspark
    major = int(pyspark.__version__.split(".")[0])
    rt = ("fabric-2.0" if major >= 4 else "fabric-1.3") if fab else ("oss-4.x" if major >= 4 else "oss-3.5")
    return fab, rt

IN_FABRIC, DETECTED = detect()
RT = DETECTED if RUNTIME == "auto" else RUNTIME
NEE_AVAILABLE = IN_FABRIC
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
print(f"fabric={IN_FABRIC} runtime={RT} nee_available={NEE_AVAILABLE} run={RUN_ID}")
if not NEE_AVAILABLE:
    print("NOTE: NEE is Fabric-only. Locally the harness runs both passes on JVM Spark, so the")
    print("      delta will be ~0 - which validates the MEASUREMENT, not the acceleration.")

fabric=False runtime=oss-3.5 nee_available=False run=20260804T121801Z
NOTE: NEE is Fabric-only. Locally the harness runs both passes on JVM Spark, so the
      delta will be ~0 - which validates the MEASUREMENT, not the acceleration.


In [3]:
from pyspark.sql import SparkSession, functions as F

# --- Session: Fabric is the default target -------------------------------------
# In Fabric you do NOT create a Spark session. The Livy layer starts it before your first
# cell runs, and `spark` (plus `sc`, `notebookutils`) are already bound. Calling
# SparkSession.builder there is at best a no-op via getOrCreate() and at worst misleading:
# master(), Delta wiring and executor shape are all decided by the Environment/pool, not here.
#
# Session-start settings belong in a %%configure -f cell ABOVE this one, or in the
# Environment's Spark properties. Only runtime-mutable keys can be set from code.
try:
    spark                      # noqa: F821  <- Fabric (and any live session): already provided
    IN_FABRIC = True
except NameError:
    # Local/dev fallback ONLY. Never runs in Fabric.
    IN_FABRIC = False
    from pyspark.sql import SparkSession
    from delta import configure_spark_with_delta_pip
    _b = (SparkSession.builder.appName(NOTEBOOK_NAME).master("local[4]")
          .config("spark.driver.memory", "2g")
          .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
          .config("spark.sql.catalog.spark_catalog",
                  "org.apache.spark.sql.delta.catalog.DeltaCatalog"))
    spark = configure_spark_with_delta_pip(_b).getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
print(("Fabric session (provided)" if IN_FABRIC else "local session (dev fallback)"),
      "| Spark", spark.version)


26/08/04 12:18:03 WARN Utils: Your hostname, vm resolves to a loopback address: 127.0.0.1; using 192.0.2.2 instead (on interface eth0)
26/08/04 12:18:03 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/usr/local/lib/python3.12/dist-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-0cf2a6f0-2903-4eac-aaa4-3c62fa1d293a;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.2.0 in central
	found io.delta#delta-storage;3.2.0 in central


	found org.antlr#antlr4-runtime;4.9.3 in central
:: resolution report :: resolve 230ms :: artifacts dl 13ms
	:: modules in use:
	io.delta#delta-spark_2.12;3.2.0 from central in [default]
	io.delta#delta-storage;3.2.0 from central in [default]
	org.antlr#antlr4-runtime;4.9.3 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   3   |   0   |   0   |   0   ||   3   |   0   |
	---------------------------------------------------------------------
:: retrieving :: org.apache.spark#spark-submit-parent-0cf2a6f0-2903-4eac-aaa4-3c62fa1d293a
	confs: [default]
	0 artifacts copied, 3 already retrieved (0kB/8ms)


26/08/04 12:18:04 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


local session (dev fallback) | Spark 3.5.1


## 1 — Representative workload (fact + dimension, the shape most ETL actually is)

In [4]:
import shutil
shutil.rmtree(TABLES_ROOT, ignore_errors=True)
FACT, DIM = f"{TABLES_ROOT}/fact_orders", f"{TABLES_ROOT}/dim_customer"

(spark.range(0, 300_000)
  .withColumn("customer_id", (F.col("id") % 4000).cast("int"))
  .withColumn("amount", F.round(F.rand()*500, 2))
  .withColumn("qty", (F.col("id") % 17 + 1).cast("int"))
  .withColumn("status", F.when(F.col("id") % 9 == 0, "cancelled").otherwise("complete"))
  .withColumn("order_date", F.date_add(F.lit("2026-01-01"), (F.col("id") % 200).cast("int")))
  .write.format("delta").mode("overwrite").save(FACT))
(spark.range(0, 4000)
  .withColumnRenamed("id","customer_id")
  .withColumn("segment", F.when(F.col("customer_id") % 3 == 0, "enterprise").otherwise("smb"))
  .withColumn("region", F.concat(F.lit("R"), (F.col("customer_id") % 12).cast("string")))
  .write.format("delta").mode("overwrite").save(DIM))
print("fact:", spark.read.format("delta").load(FACT).count(),
      "| dim:", spark.read.format("delta").load(DIM).count())

fact: 300000 | dim: 4000


In [5]:
# Query set spanning the shapes that matter - including two DESIGNED to trigger fallback,
# so the report shows contrast rather than a uniform green wall.
from pyspark.sql.types import StringType

QUERIES = {
  "agg_scan_heavy": lambda: (spark.read.format("delta").load(FACT)
      .where("status = 'complete'")
      .groupBy("order_date").agg(F.sum("amount").alias("rev"), F.count("*").alias("n"))),
  "join_fact_dim": lambda: (spark.read.format("delta").load(FACT).alias("f")
      .join(F.broadcast(spark.read.format("delta").load(DIM).alias("d")), "customer_id")
      .groupBy("segment","region").agg(F.sum("amount").alias("rev"))),
  "window_rank": lambda: (spark.read.format("delta").load(FACT)
      .withColumn("rn", F.row_number().over(
          __import__("pyspark").sql.Window.partitionBy("customer_id").orderBy(F.col("order_date").desc())))
      .where("rn = 1").groupBy("status").count()),
  "python_udf_FALLBACK": lambda: (spark.read.format("delta").load(FACT)
      .withColumn("clean", F.udf(lambda s: (s or "").upper(), StringType())("status"))
      .groupBy("clean").agg(F.sum("amount").alias("rev"))),
  "nested_types_FALLBACK": lambda: (spark.read.format("delta").load(FACT)
      .withColumn("payload", F.to_json(F.struct("customer_id","amount","status")))
      .withColumn("back", F.from_json("payload", "customer_id INT, amount DOUBLE, status STRING"))
      .select(F.col("back.status").alias("s"), "amount")
      .groupBy("s").agg(F.avg("amount").alias("avg_amt"))),
}
print("queries:", list(QUERIES))

queries: ['agg_scan_heavy', 'join_fact_dim', 'window_rank', 'python_udf_FALLBACK', 'nested_types_FALLBACK']


### 1b — Domain-shaped workloads (pension-fund / investment-data patterns)
The generic set above isolates *mechanisms*. This set mirrors the shapes that actually dominate an
investment-data platform: a large position/transaction fact joined to slowly-changing dimensions, an
incremental silver merge driven by a watermark, a BI-serving gold aggregate, and a
point-in-time/as-of lookup. Swap `QUERY_SET = "domain"` to measure these instead — or `"both"`.

In [6]:

QUERY_SET = "both"      # generic | domain | both

# --- domain fixtures: positions fact + instrument dim + as-of valuation ---
POS, INSTR, VAL = f"{TABLES_ROOT}/fact_positions", f"{TABLES_ROOT}/dim_instrument", f"{TABLES_ROOT}/fact_valuation"
(spark.range(0, 300_000)
  .withColumn("instrument_id", (F.col("id") % 3000).cast("int"))
  .withColumn("portfolio_id", (F.col("id") % 40).cast("int"))
  .withColumn("quantity", F.round(F.rand()*10000, 2))
  .withColumn("book_value", F.round(F.rand()*1_000_000, 2))
  .withColumn("asset_class", F.when(F.col("id") % 5 == 0, "equity")
                              .when(F.col("id") % 5 == 1, "fixed_income")
                              .when(F.col("id") % 5 == 2, "property")
                              .otherwise("alternatives"))
  .withColumn("position_date", F.date_add(F.lit("2026-01-01"), (F.col("id") % 180).cast("int")))
  .write.format("delta").mode("overwrite").save(POS))
(spark.range(0, 3000).withColumnRenamed("id","instrument_id")
  .withColumn("sector", F.concat(F.lit("SEC"), (F.col("instrument_id") % 24).cast("string")))
  .withColumn("currency", F.when(F.col("instrument_id") % 3 == 0, "GBP")
                           .when(F.col("instrument_id") % 3 == 1, "USD").otherwise("EUR"))
  .withColumn("is_active", F.col("instrument_id") % 17 != 0)
  .write.format("delta").mode("overwrite").save(INSTR))
(spark.range(0, 60_000)
  .withColumn("instrument_id", (F.col("id") % 3000).cast("int"))
  .withColumn("valuation_date", F.date_add(F.lit("2026-01-01"), (F.col("id") % 20).cast("int")))
  .withColumn("price", F.round(F.rand()*500 + 10, 4))
  .write.format("delta").mode("overwrite").save(VAL))

from pyspark.sql import Window

DOMAIN_QUERIES = {
  # 1. Fact x dimension join with filtering - the bread and butter of exposure reporting
  "dom_exposure_by_sector": lambda: (
      spark.read.format("delta").load(POS).alias("p")
        .where("position_date >= '2026-03-01'")
        .join(F.broadcast(spark.read.format("delta").load(INSTR).alias("i")), "instrument_id")
        .where("is_active")
        .groupBy("asset_class", "sector", "currency")
        .agg(F.sum("book_value").alias("exposure"), F.countDistinct("instrument_id").alias("instruments"))),

  # 2. Incremental silver merge shape - watermark-filtered source, dedup to latest per key
  "dom_incremental_silver": lambda: (
      spark.read.format("delta").load(POS)
        .where("position_date >= '2026-06-01'")                       # watermark slice
        .withColumn("rn", F.row_number().over(
            Window.partitionBy("portfolio_id", "instrument_id").orderBy(F.col("position_date").desc())))
        .where("rn = 1")
        .select("portfolio_id", "instrument_id", "quantity", "book_value", "position_date")),

  # 3. BI-serving gold aggregate - multi-grain rollup feeding Direct Lake
  "dom_gold_rollup": lambda: (
      spark.read.format("delta").load(POS).alias("p")
        .join(spark.read.format("delta").load(INSTR).alias("i"), "instrument_id")
        .groupBy("position_date", "asset_class", "currency")
        .agg(F.sum("book_value").alias("total_value"),
             F.avg("book_value").alias("avg_value"),
             F.count("*").alias("position_count"))),

  # 4. As-of / point-in-time valuation join - the shape that punishes bad key types
  "dom_asof_valuation": lambda: (
      spark.read.format("delta").load(POS).alias("p")
        .join(spark.read.format("delta").load(VAL).alias("v"),
              (F.col("p.instrument_id") == F.col("v.instrument_id")) &
              (F.col("v.valuation_date") <= F.col("p.position_date")))
        .groupBy("p.portfolio_id")
        .agg(F.sum(F.col("p.quantity") * F.col("v.price")).alias("market_value"))),
}

if QUERY_SET == "domain":
    QUERIES = DOMAIN_QUERIES
elif QUERY_SET == "both":
    QUERIES = {**QUERIES, **DOMAIN_QUERIES}
print(f"query set '{QUERY_SET}':", list(QUERIES))
print("\nNote: dom_asof_valuation uses an INEQUALITY join condition - expect")
print("BroadcastNestedLoopJoin or a range join in the plan (analyzer P002). It is included")
print("deliberately: as-of joins are common in investment data and are a known performance trap.")


query set 'both': ['agg_scan_heavy', 'join_fact_dim', 'window_rank', 'python_udf_FALLBACK', 'nested_types_FALLBACK', 'dom_exposure_by_sector', 'dom_incremental_silver', 'dom_gold_rollup', 'dom_asof_valuation']

Note: dom_asof_valuation uses an INEQUALITY join condition - expect
BroadcastNestedLoopJoin or a range join in the plan (analyzer P002). It is included
deliberately: as-of joins are common in investment data and are a known performance trap.


## 2 — The measurement harness
Plan composition is read straight from the formatted plan: native operators carry `*Transformer` /
`*NativeFileScan` suffixes, and `VeloxColumnarToRowExec` marks each conversion boundary. Coverage =
native / (native + conversions). Wall-clock takes the **minimum** of repeated runs to blunt noise.

In [7]:
def plan_text(df):
    return df._jdf.queryExecution().explainString(
        df._sc._jvm.org.apache.spark.sql.execution.ExplainMode.fromString("formatted"))

NATIVE_RE = re.compile(r"Transformer|NativeFileScan")
CONV_RE   = re.compile(r"VeloxColumnarToRowExec|RowToVeloxColumnar")

def plan_stats(p):
    native, conv = len(NATIVE_RE.findall(p)), len(CONV_RE.findall(p))
    total = native + conv
    return {"native_ops": native, "conversions": conv,
            "coverage_pct": round(100.0*native/total,1) if total else 0.0}

def timed(fn, repeats=REPEATS):
    times = []
    for _ in range(repeats):
        t0 = time.time(); fn().collect(); times.append(time.time()-t0)
    return round(min(times), 3)

def set_nee(enabled: bool):
    """NEE is a session-start setting in Fabric (Environment > Acceleration, or %%configure).
    spark.conf.set works for the flag on some builds but is NOT guaranteed - the honest approach
    is to run this notebook twice under two Environments and compare reports."""
    try:
        spark.conf.set("spark.native.enabled", "true" if enabled else "false")
        return spark.conf.get("spark.native.enabled") == ("true" if enabled else "false")
    except Exception as e:
        print(f"  (could not toggle NEE at runtime: {type(e).__name__} - use two Environments)")
        return False
print("harness ready")

harness ready


In [8]:
results = []
for mode, want_nee in [("nee_on", True), ("nee_off", False)]:
    applied = set_nee(want_nee) if NEE_AVAILABLE else False
    for name, builder in QUERIES.items():
        df = builder()
        p = plan_text(df)
        st = plan_stats(p)
        secs = timed(builder)
        results.append({"run_id": RUN_ID, "mode": mode, "nee_applied": applied, "query": name,
                        "secs": secs, **st})
        print(f"{mode:8s} {name:22s} {secs:7.3f}s  native={st['native_ops']:3d} "
              f"conv={st['conversions']:3d} coverage={st['coverage_pct']}")
print(f"\n{len(results)} measurements")

nee_on   agg_scan_heavy           1.958s  native=  0 conv=  0 coverage=0.0


nee_on   join_fact_dim            2.667s  native=  0 conv=  0 coverage=0.0


nee_on   window_rank              2.875s  native=  0 conv=  0 coverage=0.0


nee_on   python_udf_FALLBACK      5.022s  native=  0 conv=  0 coverage=0.0


nee_on   nested_types_FALLBACK    3.846s  native=  0 conv=  0 coverage=0.0


...

nee_off  nested_types_FALLBACK    2.624s  native=  0 conv=  0 coverage=0.0


nee_off  dom_exposure_by_sector   2.326s  native=  0 conv=  0 coverage=0.0


nee_off  dom_incremental_silver   1.299s  native=  0 conv=  0 coverage=0.0


nee_off  dom_gold_rollup          1.818s  native=  0 conv=  0 coverage=0.0


nee_off  dom_asof_valuation       2.534s  native=  0 conv=  0 coverage=0.0

18 measurements


## 3 — The report: is NEE earning its keep?

In [9]:
by_q = {}
for r in results:
    by_q.setdefault(r["query"], {})[r["mode"]] = r

print(f"{'query':24s} {'NEE on':>9s} {'NEE off':>9s} {'delta':>9s}  verdict")
print("-"*78)
rows = []
for q, m in by_q.items():
    on, off = m.get("nee_on"), m.get("nee_off")
    if not (on and off): continue
    delta = off["secs"] - on["secs"]
    pct = (delta/off["secs"]*100) if off["secs"] else 0
    if not NEE_AVAILABLE:
        verdict = "n/a locally (both JVM)"
    elif on["coverage_pct"] in (None, 0):
        verdict = "FULL FALLBACK - investigate or disable NEE"
    elif pct > 10:
        verdict = f"NEE faster by {pct:.0f}%"
    elif pct < -10:
        verdict = f"NEE SLOWER by {-pct:.0f}% - conversion overhead"
    else:
        verdict = "no material difference"
    print(f"{q:24s} {on['secs']:8.3f}s {off['secs']:8.3f}s {delta:+8.3f}s  {verdict}")
    rows.append({**on, "secs_nee_off": off["secs"], "delta_secs": round(delta,3),
                 "pct_gain": round(pct,1), "verdict": verdict})

os.makedirs(os.path.dirname(REPORT_PATH), exist_ok=True)
from pyspark.sql.types import StructType, StructField, StringType as S, DoubleType as D, LongType as L, BooleanType as B
schema = StructType([StructField("run_id",S()),StructField("mode",S()),StructField("nee_applied",B()),
    StructField("query",S()),StructField("secs",D()),StructField("native_ops",L()),
    StructField("conversions",L()),StructField("coverage_pct",D()),StructField("secs_nee_off",D()),
    StructField("delta_secs",D()),StructField("pct_gain",D()),StructField("verdict",S())])
ordered = [tuple(r[f.name] for f in schema.fields) for r in rows]
spark.createDataFrame(ordered, schema).write.format("delta").mode("append").option("mergeSchema","true").save(REPORT_PATH)
print(f"\nreport appended to {REPORT_PATH} (versioned by run_id - trend it over time)")
assert len(rows) == len(QUERIES)

query                       NEE on   NEE off     delta  verdict
------------------------------------------------------------------------------
agg_scan_heavy              1.958s    1.065s   -0.893s  n/a locally (both JVM)
join_fact_dim               2.667s    1.645s   -1.022s  n/a locally (both JVM)
window_rank                 2.875s    1.521s   -1.354s  n/a locally (both JVM)
python_udf_FALLBACK         5.022s    2.124s   -2.898s  n/a locally (both JVM)
nested_types_FALLBACK       3.846s    2.624s   -1.222s  n/a locally (both JVM)
dom_exposure_by_sector      2.759s    2.326s   -0.433s  n/a locally (both JVM)
dom_incremental_silver      2.118s    1.299s   -0.819s  n/a locally (both JVM)
dom_gold_rollup             2.139s    1.818s   -0.321s  n/a locally (both JVM)
dom_asof_valuation          2.318s    2.534s   +0.216s  n/a locally (both JVM)



report appended to /tmp/fabric_nee_demo/_ops/nee_report (versioned by run_id - trend it over time)


In [10]:
# Which queries would fall back, and why - static prediction cross-checked against the plans.
import sys; # --- Importing the toolkit modules in Fabric -----------------------------------
# `sys.path.insert(0, os.getcwd())` works locally but NOT in Fabric - there is no local
# working directory holding your .py files. Fabric options, in order of robustness:
#   1. Environment custom library  - upload a .whl (or .py) to the Environment and Publish.
#                                    Survives across notebooks; the production choice.
#   2. Notebook Resources (builtin) - upload the .py to the notebook's Resources folder,
#                                    then `from builtin import fabric_workload_advisor`.
#   3. Lakehouse Files + sys.path   - upload to Files/code, then:
#                                    sys.path.append("/lakehouse/default/Files/code")
#   4. %run another notebook        - for notebook-defined helpers (not .py modules).
# Python (non-Spark) notebooks currently support only the wheel route:
#   %pip install /lakehouse/default/Files/code/toolkit-0.1-py3-none-any.whl
import sys, os
for _p in ("/lakehouse/default/Files/code", os.getcwd()):
    if os.path.isdir(_p) and _p not in sys.path:
        sys.path.append(_p)
try:
    from spark_plan_analyzer import analyze_plan_nee, lint_nee
    for name, builder in QUERIES.items():
        f = analyze_plan_nee(plan_text(builder()), name)
        if f:
            for x in f: print(f"[{x.code}] {name}: {x.message}")
    print("\nStatic source-level prediction (what to fix before it ever runs):")
    src = "F.udf(lambda s: s.upper())\nfrom_json(payload)"
    for x in lint_nee(src, "example", RT if RT.startswith("fabric") else "fabric-1.3"):
        print(f"[{x.code}] {x.message}\n     -> {x.suggestion}")
except ImportError:
    print("spark_plan_analyzer.py not on the path - copy it beside this notebook (or add to the Environment).")


Static source-level prediction (what to fix before it ever runs):
[N002] Python UDF in the plan - a single unsupported expression drops its whole enclosing operator to JVM execution, adding columnar-to-row conversion at the boundary.
     -> Replace with built-ins where possible. NEE on Runtime 2.0 added Python/Scala UDF support, but built-ins remain the only way to keep the operator fully native AND Catalyst-visible.
[N005] Nested/complex type manipulation - deeply nested struct/map operations are a common fallback trigger.
     -> Flatten to columnar operations where the logic allows; do the nesting once at the edge rather than repeatedly in the hot path.


## 4 — How to run this properly in Fabric
1. Create **two Environments** identical but for the Acceleration setting (NEE on / off) — or one
   Environment plus a `%%configure` cell per run. Runtime-toggling `spark.native.enabled` mid-session
   is not guaranteed; two Environments is the trustworthy comparison.
2. Run this notebook against each, with `TABLES_ROOT` pointing at a Lakehouse path and a query set
   swapped for **your** real workload shapes.
3. On **Runtime 2.0**, run a third pass with `spark.sql.ansi.enabled=false` — since NEE falls back
   under ANSI, this is the pass that reveals whether your 2.0 environment is getting native execution
   at all. Compare all three.
4. Trend `nee_report` over time: coverage regressions usually mean someone added a UDF.

**Interpreting results honestly:** an I/O-bound query showing no gain is not a failure — Microsoft
is explicit that NEE targets compute-intensive work. The result to act on is *negative* delta with
low coverage: that is conversion overhead without native execution, and either the trigger gets fixed
or NEE gets disabled for that job.

In [11]:
print("artifacts:", sorted(os.listdir(TABLES_ROOT)))
spark.stop(); print("session stopped")

artifacts: ['_ops', 'dim_customer', 'dim_instrument', 'fact_orders', 'fact_positions', 'fact_valuation']


session stopped
